# Modelo de Probabilidad de Compra por Marca

## Proceso de Modelado

 1. **Selección de Marca**

Marca seleccionada: **Marca 0**

`SELECTED_BRAND`

### 2. **Creación de Variable Target**
**Target = 1**: Cliente ha comprado la marca seleccionada al menos una vez (55%)

**Target = 0**: Cliente nunca ha comprado la marca (45%)
- Distribución: 55% compradores, 45% no compradores (balanceado)

### 3. **Variables Predictoras (Features)**

#### **Variables RFM (Recency, Frequency, Monetary)**
- `recency_days`: Días desde última compra (comportamiento reciente)
- `frequency`: Número total de compras (lealtad)
- `monetary`: Valor monetario total (poder adquisitivo)
- `avg_ticket`: Ticket promedio por compra
- `r_score`, `f_score`, `m_score`: Scores RFM (1-3 cada uno)
- `rfm_total`: Score RFM total (suma de r+f+m, rango 3-9)

#### **Variables de Engagement**
- `tenure_days`: Antigüedad como cliente
- `active_days`: Días con compras activas

#### **Variables de Diversidad de Compra**
- `brand_variety`: Número de marcas diferentes compradas
- `sku_variety`: Número de SKUs diferentes comprados
- `wallet_hhi`: Índice Herfindahl de concentración (0-1, donde 1 = monógamo)
- `top_brand_share`: % de compras en su marca principal
- `brands_bought`: Total de marcas compradas

#### **Variables Categóricas (One-Hot Encoded)**
- **Segmento RFM**: Campeones, Leales, Nuevos, Potenciales leales, Prometedores, Necesitan atención, En riesgo, Hibernando, Otros
- **Brand Affinity**: Monógamo (leal a 1 marca), Explorador (compra muchas marcas), Mixto

**Total: 28 features**

### 4. **División Train/Test**
- 75% train (3,159 clientes) / 25% test (1,054 clientes)
- **Split estratificado**: Mantiene la misma proporción de compradores en train y test

### 5. **Modelo**
- **XGBoost Classifier** (configurado en Cell 5)
- Optimizado para clasificación binaria
- Incluye `scale_pos_weight` para balancear clases

### 6. **Evaluación**
- AUC-ROC, Precision, Recall, F1-Score
- Matriz de confusión
- Feature importance

---

## Variables Relevantes Esperadas

Basándose en el tipo de problema, las variables más relevantes probablemente serán:

1. **`rfm_total` y scores individuales**: Clientes de alto valor (RFM alto) suelen comprar marcas premium
2. **`brand_variety` y `wallet_hhi`**: Indica si el cliente es monógamo o explorador
3. **Segmento RFM**: Los "Campeones" y "Leales" probablemente compran más la marca
4. **`top_brand_share`**: Si su marca principal coincide con Marca 0
5. **`frequency` y `monetary`**: Clientes frecuentes y de alto gasto

*La importancia real se verificará después del entrenamiento en Cell 8.*

### Model Setup


Marca seleccionada: **Marca 0**

`SELECTED_BRAND`

In [0]:
# DBTITLE 1,Parameters
CATALOG = "workspace"             # <-- ajustar
SCHEMA  = "default"              # <-- ajustar
TABLE   = "customer_profiles"
PURCHASE_TABLE = "prueba_bavaria_compras"

# Marca seleccionada para modelar probabilidad de compra
SELECTED_BRAND = "Marca 0"  # <-- Cambiar si deseas otra marca (Marca 0, Marca 1, Marca 2, Marca 3)

TEST_SIZE    = 0.25
RANDOM_STATE = 42

FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"
FULL_PURCHASE_TABLE = f"{CATALOG}.{SCHEMA}.{PURCHASE_TABLE}"
print(f"Tabla clientes: {FULL_TABLE}")
print(f"Tabla compras: {FULL_PURCHASE_TABLE}")
print(f"Marca objetivo: {SELECTED_BRAND}")

### **Creación de Variable Target**
**Target = 1**: Cliente ha comprado la marca seleccionada al menos una vez (55%)

**Target = 0**: Cliente nunca ha comprado la marca (45%)
- Distribución: 55% compradores, 45% no compradores (balanceado)

In [0]:
# DBTITLE 1,Cargar datos y crear target variable
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, trim, upper

# Cargar perfil de clientes
sdf_customers = spark.table(FULL_TABLE)
print(f"Customer profiles: {sdf_customers.count():,} clientes, {len(sdf_customers.columns)} columnas")

# Cargar datos de compras y limpiar nombres de marca
sdf_purchases = (spark.table(FULL_PURCHASE_TABLE)
                 .withColumn("brand_clean", trim(upper(col("BRAND"))))
                 .select("CUSTOMER", "brand_clean"))

# Identificar clientes que compraron la marca seleccionada
selected_brand_clean = SELECTED_BRAND.strip().upper()
customers_bought_brand = (sdf_purchases
                          .filter(col("brand_clean") == selected_brand_clean)
                          .select("CUSTOMER")
                          .distinct())

print(f"\nClientes que compraron '{SELECTED_BRAND}': {customers_bought_brand.count():,}")

# Crear dataset con target: 1 si compró la marca, 0 si no
from pyspark.sql.functions import when, lit

sdf_with_target = (sdf_customers
                   .join(customers_bought_brand.withColumn("bought_brand", lit(1)),
                         on=sdf_customers.customer == customers_bought_brand.CUSTOMER,
                         how="left")
                   .withColumn("target", when(col("bought_brand") == 1, 1).otherwise(0))
                   .drop("bought_brand", "CUSTOMER"))

# Convertir a pandas
df = sdf_with_target.toPandas()

print(f"\nDataset final: {len(df):,} clientes")
print(f"Distribución del target:")
print(df['target'].value_counts().sort_index())
print(f"Tasa de compra: {df['target'].mean():.2%}")
df.head()

### **Variables Predictoras (Features)**



In [0]:
# DBTITLE 1,Preparar matriz de features (X) y target (y)
# Columnas a excluir de features (identificadores y target)
DROP_COLS = ['customer', 'rfm_code', 'top_brand', 'rfm_segment', 'brand_affinity', 'target']

# Encoding simple para la marca principal (top_brand que más compra el cliente)
df['brand_code'] = df['top_brand'].astype('category').cat.codes

# One-hot encoding para rfm_segment (Campeones, Leales, etc.)
df = pd.concat([df, pd.get_dummies(df['rfm_segment'], prefix='seg')], axis=1)

# One-hot encoding para brand_affinity (Monogamo, Explorador, Mixto)
df = pd.concat([df, pd.get_dummies(df['brand_affinity'], prefix='affinity')], axis=1)

# Crear matriz de features X (solo columnas numéricas)
X = (df.drop(columns=[c for c in DROP_COLS if c in df.columns])
       .select_dtypes(include=[np.number, 'bool']))

# Crear variable target y
y = df['target'].values

print(f"X shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"\ny (target) shape: {y.shape}")
print(f"Clientes que compraron {SELECTED_BRAND}: {y.sum():,} ({y.mean():.2%})")
print(f"Clientes que NO compraron {SELECTED_BRAND}: {(1-y).sum():,} ({(1-y.mean()):.2%})")

### 4. **División Train/Test**
- 75% train (3,159 clientes) / 25% test (1,054 clientes)
- **Split estratificado**: Mantiene la misma proporción de compradores en train y test

In [0]:
# DBTITLE 1,Split estratificado train / test
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,  # Mantiene la misma proporción de compradores en train y test
)
print(f"Train set: {len(X_tr):,} clientes")
print(f"Test set:  {len(X_te):,} clientes")
print(f"\nCompradores en train: {y_tr.sum():,} ({y_tr.mean():.2%})")
print(f"Compradores en test:  {y_te.sum():,} ({y_te.mean():.2%})")

In [0]:
%pip install xgboost

### **Model**


In [0]:
# DBTITLE 1,Entrenar XGBoost
import xgboost as xgb

pos_weight = (1 - y_tr.mean()) / y_tr.mean()

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    scale_pos_weight=pos_weight,
)

model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
print("Entrenamiento completado.")


### **Evaluación**
- AUC-ROC, Precision, Recall, F1-Score
- Matriz de confusión
- Feature importance

In [0]:
# DBTITLE 1,Métricas: AUC, Precision, Recall
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, confusion_matrix, classification_report,
)

proba = model.predict_proba(X_te)[:, 1]
auc   = roc_auc_score(y_te, proba)

# Umbral por defecto 0.50
pred_05 = (proba >= 0.5).astype(int)
p_05 = precision_score(y_te, pred_05)
r_05 = recall_score(y_te, pred_05)
f_05 = f1_score(y_te, pred_05)

# Umbral óptimo por F1
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_te, proba)
f1s = 2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-9)
best_idx = int(np.nanargmax(f1s[:-1]))
t_opt = float(thr_curve[best_idx])

pred_opt = (proba >= t_opt).astype(int)
p_opt = precision_score(y_te, pred_opt)
r_opt = recall_score(y_te, pred_opt)
f_opt = f1_score(y_te, pred_opt)

metrics = pd.DataFrame({
    'metric':    ['AUC-ROC',
                  'Precision @ 0.50', 'Recall @ 0.50', 'F1 @ 0.50',
                  f'Precision @ opt({t_opt:.3f})',
                  f'Recall @ opt({t_opt:.3f})',
                  'F1 óptimo'],
    'value':     [auc, p_05, r_05, f_05, p_opt, r_opt, f_opt],
})
display(metrics)

### **Confusion Matrix and Classification Report**

In [0]:

# DBTITLE 1,Matriz de confusión y reporte detallado
print("Matriz de confusión @ 0.50")
print(confusion_matrix(y_te, pred_05))
print()
print("Classification report @ 0.50")
print(classification_report(y_te, pred_05, digits=4))
print()
print(f"Matriz de confusión @ umbral óptimo ({t_opt:.3f})")
print(confusion_matrix(y_te, pred_opt))

### **Top Important Features**

In [0]:
# DBTITLE 1,Top features por importancia
importances = (pd.Series(model.feature_importances_, index=X.columns)
                 .sort_values(ascending=False)
                 .head(20)
                 .rename('importance')
                 .to_frame())
importances['variable'] = importances.index
display(importances[['variable', 'importance']])

# Resultados y Conclusiones

## Rendimiento del Modelo

El modelo XGBoost entrenado muestra **excelente desempeño** para predecir la probabilidad de compra de **Marca 0**:

| Métrica | Valor |
|---------|-------|
| **AUC-ROC** | **95.4%** |
| Precision @ 0.50 | 88.7% |
| Recall @ 0.50 | 90.7% |
| F1-Score @ 0.50 | 89.7% |
| F1-Score Óptimo | 90.0% |

**Interpretación:**
- **AUC-ROC = 95.4%**: El modelo discrimina extremadamente bien entre compradores y no compradores
- **Alta Precision (88.7%)**: De los clientes que el modelo predice que comprarán, el 88.7% realmente lo hacen
- **Alto Recall (90.7%)**: El modelo identifica correctamente al 90.7% de los compradores reales

---

## Variables Más Relevantes

Las **top 10 variables** que más influyen en la probabilidad de compra de Marca 0 son:

| Ranking | Variable | Importancia | Interpretación |
|---------|----------|------------|----------------|
| **1** | **`brand_code`** | **37.1%** | **Qué marca es la favorita del cliente** |
| 2 | `brand_variety` | 9.7% | Número de marcas diferentes que compra |
| 3 | `wallet_hhi` | 7.1% | Índice de concentración (1 = monógamo) |
| 4 | `frequency` | 4.5% | Número total de compras |
| 5 | `f_score` | 4.0% | Score de frecuencia RFM (1-3) |
| 6 | `active_days` | 2.5% | Días con actividad de compra |
| 7 | `top_brand_share` | 2.4% | % de compras en su marca principal |
| 8 | `rfm_total` | 2.0% | Score RFM total (3-9) |
| 9 | `brands_bought` | 1.8% | Total de marcas diferentes compradas |
| 10 | `affinity_Monogamo` | 1.8% | Si el cliente es monógamo a una marca |

---

## Insights Clave

### 1️⃣ **La marca favorita actual es el predictor más fuerte (37%)**
➡️ Si Marca 0 ya es la marca principal del cliente (`top_brand`), la probabilidad de compra es muy alta.

**Implicación de negocio:** Foco en **retención** de clientes actuales de Marca 0 - son el segmento más valioso.

### 2️⃣ **La variedad de marca importa (9.7%)**
➡️ Clientes que compran **muchas marcas diferentes** tienen menor probabilidad de comprar Marca 0.

**Implicación de negocio:** Clientes "exploradores" requieren estrategias de **conversión y prueba**, mientras que clientes monógamos requieren **fidelización**.

### 3️⃣ **La concentración de wallet (HHI) es clave (7.1%)**
➡️ Clientes con **alto HHI** (monógamos) son más predecibles - si su marca favorita es Marca 0, comprarán; si no, no comprarán.

**Implicación de negocio:** Identificar clientes monógamos de **otras marcas** para campañas de **switching**.

### 4️⃣ **La frecuencia y engagement importan (4.5% + 2.5%)**
➡️ Clientes **frecuentes y activos** tienen mayor probabilidad de comprar.

**Implicación de negocio:** Programas de **lealtad y recompensas** pueden aumentar frecuencia y, por ende, probabilidad de compra de Marca 0.

---

## Recomendaciones de Negocio

1. **Segmentación de clientes:**
   - **Alta probabilidad (>70%)**: Clientes actuales de Marca 0 → Campañas de retención
   - **Media probabilidad (30-70%)**: Clientes mixtos → Promoción y prueba de producto
   - **Baja probabilidad (<30%)**: Clientes leales a otras marcas → Campañas de switching agresivas

2. **Enfoque en retención**: La mayoría de compradores de Marca 0 ya son clientes leales. Proteger esta base es crítico.

3. **Activación de clientes inactivos**: Clientes con baja frecuencia pero que históricamente compraron Marca 0 son oportunidades de reactivación.

4. **Cross-selling**: Clientes con alta variedad de marca son menos leales pero más abiertos a probar - ofrecer bundles o descuentos.

---

## Siguiente Paso

Ejecutar Cell 9 para **scoring del universo completo** y guardar las probabilidades de compra de todos los clientes en una tabla Delta.

In [0]:
# DBTITLE 1,Scoring del universo completo y escritura en Delta
# Genera la probabilidad de compra de la marca seleccionada para todos los clientes
df['purchase_proba'] = model.predict_proba(X)[:, 1]
df['predicted_label_05']  = (df['purchase_proba'] >= 0.5).astype(int)
df['predicted_label_opt'] = (df['purchase_proba'] >= t_opt).astype(int)

scored = df[['target', 'top_brand',
             'purchase_proba', 'predicted_label_05', 'predicted_label_opt',
             'rfm_segment', 'brand_affinity']]

sdf_out = spark.createDataFrame(scored)
OUTPUT_TABLE = f"{CATALOG}.{SCHEMA}.purchase_proba_{SELECTED_BRAND.lower().replace(' ', '_')}"
(sdf_out.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(OUTPUT_TABLE))
print(f"Scores guardados en {OUTPUT_TABLE}")
print(f"\nTabla contiene probabilidad de compra para: {SELECTED_BRAND}")